# 10 — DistilBERT supervised fine-tuning

New **text-only Transformer classifier**, not a modification of LightGBM. All DistilBERT encoder weights and a new binary classification head are trained. This differs from notebook 07, which froze MiniLM and trained logistic regression on its embeddings.

## Start here: M1 / 8 GB

This is **not a lightweight TF-IDF experiment**. The default is one saved outer fold, not 25, but nested evaluation still involves **eight neural training phases** (three inner fits plus the outer fit, each with stopping and fresh refit). Small batches reduce peak memory; fewer folds primarily reduce total time. MPS uses the Mac's shared memory, so avoid other heavy notebooks/apps. Memory fit and runtime are not guaranteed.

1. Select the existing project `.venv` kernel. Run the dependency cell with installation disabled first; dependencies may already be installed.
2. Run setup and configuration. Check reported memory, device, frozen data and the **same-fold** saved A2 reference.
3. Set `PREPARE_TOKENS=True` and run its cell. The first use downloads the pinned pretrained checkpoint's tokenizer; the model weights are downloaded only on the first resource check or training call. Internet is required for first downloads, not for cached offline training.
4. Recommended: set `RUN_RESOURCE_CHECK=True` for three optimizer updates on training rows only. This is not a model-performance test.
5. If resources are acceptable, reset that flag to `False`, set `TRAIN_MODEL=True`, then run the training and comparison cells. Keep `OUTER_LIMIT=1`. All rows of that fixed outer training/validation split are used, not a reduced evaluation sample.
6. Compare against A2 on **that exact outer fold**, not its 25-fold mean. A clearly disappointing result can justify stopping this candidate for practical reasons, but a single fold is not proof about overall performance. Do not adjust this fold's settings repeatedly to chase its score.

No installation, model download or training occurs unless its explicit flag is enabled. The reserved test file is never loaded. No cluster setup is required.


In [ ]:
# Run this cell once with False first. No terminal needed.
INSTALL_DEPENDENCIES = False

from pathlib import Path
import importlib.metadata
import importlib.util
import subprocess
import sys
import tempfile

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/d1_distilbert.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the complete project repository.')
if INSTALL_DEPENDENCIES:
    # Preserve baseline libraries rather than silently upgrading their fold/feature code.
    with tempfile.TemporaryDirectory(prefix='bt4012_distilbert_install_') as folder:
        constraints = Path(folder) / 'constraints.txt'
        core = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'joblib', 'threadpoolctl']
        constraints.write_text('\n'.join(f'{p}=={importlib.metadata.version(p)}' for p in core))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                               '--timeout', '120', '--retries', '5', '-c', str(constraints),
                               '-r', str(ROOT / 'requirements-10.txt')])
    print('Installation completed. Restart only this notebook kernel, then disable installation.')
else:
    for name in ['torch', 'transformers', 'psutil']:
        try:
            print(name, importlib.metadata.version(name))
        except importlib.metadata.PackageNotFoundError:
            print(name, 'MISSING — enable installation, run this cell, then restart this kernel.')


## 1. Frozen evaluation protocol and memory settings

| Component | Fixed initial choice |
|---|---|
| Model | `distilbert/distilbert-base-uncased`, immutable revision recorded in helper |
| Inputs | Title, company profile, description, requirements, benefits; text only |
| Token policy | 256-token allocation; each field has a fixed budget and name; preserve head and tail of long fields |
| Network | Pretrained DistilBERT + two-class head; all weights trainable |
| Optimizer | AdamW, learning rate 2e-5, weight decay 0.01 |
| Memory | Float32, batch 2, gradient accumulation 8 (effective batch up to 16), gradient checkpointing |
| Imbalance | Balanced loss weights calculated from each fitting subset; no SMOTE or oversampling |
| Epoch selection | At most 3 epochs, patience 1, best PR-AUC on grouped stopping rows within training; then fresh refit for the chosen epoch count |
| Threshold | Maximum F1 on three grouped inner out-of-fold predictions, never on outer validation |
| Outer folds | Restore notebook 02's saved 5×5 folds; evaluate only the first one by default |

Field budgets include their names: title 24, company profile 48, description 96, requirements 56, benefits 24; separators bring the maximum to 254. Missing fields get `missing`. Budgets are fixed before viewing outer results. Unused budgets are not reassigned, so truncation and omitted details may limit this candidate. Truncation rates are reported below.

A2 also has structured/style features; this is an **alternative text model**, not a controlled model-only replacement on identical features. Both share the same outer assessment rows. Inner and stopping folds are deterministic within the installed library version; package versions and source changes invalidate caches.


In [ ]:
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import psutil
from src import d1_distilbert as experiment

# Keep this at 1 for the initial screen. None explicitly enables all configured folds.
OUTER_LIMIT = 1
CONFIG = experiment.Config(repeats=5, outer_limit=OUTER_LIMIT,
    batch_size=2, accumulation=8, max_length=256,
    max_epochs=3, patience=1, learning_rate=2e-5,
    inner_splits=3, stopping_splits=5, seed=42, device='auto')

train, FOLDS = experiment.load_inputs(ROOT, CONFIG)
CACHE_DIR = experiment.cache_directory(ROOT, CONFIG)
memory = psutil.virtual_memory()
print('Python:', sys.executable)
print('Device:', experiment.device_name(CONFIG.device))
print(f'RAM: {memory.total / 2**30:.1f} GB total; {memory.available / 2**30:.1f} GB available now')
print(f'Development: {len(train):,} jobs; {int(train.fraudulent.sum()):,} fraudulent')
print(f'Restored {len(FOLDS)} outer folds; requesting {OUTER_LIMIT or len(FOLDS)}')
print('Environment:', experiment.versions())
print('Cache:', CACHE_DIR)

# Show the exact first-fold A2 reference before any new model training.
import pandas as pd
from IPython.display import display
baseline_path = ROOT / 'reports/tables/model_logreg_a2_fold_scores.csv'
if baseline_path.is_file():
    baseline = pd.read_csv(baseline_path)
    r, k, _, _ = FOLDS[0]
    print('Saved A2 first-fold reference (identity is checked again in the final comparison):')
    display(baseline.loc[(baseline.repeat == r) & (baseline.fold == k),
                        ['repeat', 'fold', 'pr_auc', 'precision', 'recall', 'f1']].round(4))
else:
    print('Saved A2 fold scores missing: restore them for a same-fold comparison without retraining.')
print('Final test not loaded. No model downloaded or trained.')


## 2. Prepare fixed tokenizer inputs

Tokenization uses the pretrained vocabulary and no labels, so it can be cached once for the development pool. The cache is roughly a padded 14k×256 integer array, not model activations. Every field retains a share of the input; the end of the posting is not silently discarded because the company profile was long.


In [ ]:
PREPARE_TOKENS = False

if PREPARE_TOKENS:
    token_ids, truncated, pad_id = experiment.prepare_tokens(train, CONFIG, CACHE_DIR)
    print('Token matrix:', token_ids.shape, f'{token_ids.nbytes / 2**20:.1f} MiB')
    display(pd.DataFrame({'field': experiment.FIELDS,
                          'share_truncated': truncated.mean(axis=0)}).round(4))
else:
    print('Token preparation disabled. Enable PREPARE_TOKENS when ready.')


## 3. Optional resource diagnostic (not performance)

Three optimizer updates, using only the first outer fold's training rows. Full sequence batches are selected from real training data. The model and optimizer are released afterwards. Timing includes warm-up effects, and allocated GPU memory is not peak total system memory. A successful check does not guarantee that the entire experiment will fit or be fast enough.


In [ ]:
RUN_RESOURCE_CHECK = False

if RUN_RESOURCE_CHECK:
    if 'token_ids' not in globals():
        raise RuntimeError('Prepare tokens first.')
    diagnostic = experiment.resource_check(train, token_ids, pad_id, CONFIG, FOLDS)
    display(pd.DataFrame([diagnostic]))
else:
    print('Resource diagnostic disabled. No training started.')


## 4. One-fold supervised fine-tuning

**This cell can take substantial time on M1/8 GB.** It trains three inner models and the outer model, each with stopping selection and fresh refit. These eight phases are necessary for the chosen nested protocol, even though only one outer fold is scored. Progress prints during each epoch and every 100 optimizer updates.

Caches are written after each completed inner or outer prediction phase. Re-running with unchanged settings reuses them. In-progress training has no epoch-level resume. Do not change the code, environment, tokenizer policy or settings during a run. Do not launch concurrent runs writing the same cache. No A2 or LightGBM retraining takes place.


In [ ]:
TRAIN_MODEL = False

if TRAIN_MODEL:
    if 'token_ids' not in globals():
        raise RuntimeError('Prepare tokens first.')
    result = experiment.run(ROOT, train, FOLDS, token_ids, pad_id, CONFIG, CACHE_DIR)
    print('Reports:', result['output_dir'])
else:
    print('Training disabled. Set TRAIN_MODEL=True only when ready to run the selected outer fold(s).')


## 5. Compare and decide whether to continue

PR-AUC here is average precision. Baselines are read from saved results and included only when their fold identities match. Read `comparison_sources.csv` for missing/mismatched references. For one fold there is **no corrected confidence interval** and no claim of statistically reliable uplift. Positive performance on one fold is a reason for further validation, not model adoption. A poor single-fold screen may be stopped to save computation, but does not establish a general negative result.

If you deliberately continue, set `OUTER_LIMIT=5` for the full first repeat, or `None` for all 25, then rerun configuration, token preparation and training. Completed results resume when settings/environment/code are unchanged. Do not tune against the first outer validation fold and then report that fold as untouched validation. The final test stays reserved.


In [ ]:
if 'result' in globals():
    columns = ['model', 'matched_folds', 'pr_auc', 'pr_auc_sd', 'precision', 'recall', 'f1']
    display(result['comparison'][columns].round(4))
    display(result['sources'])
    if len(result['scores']) == 1:
        print('Exploratory single-fold result only. Compare against the matched A2 row, not its 25-fold mean.')
    print('Prediction caches:', result['cache_dir'])
    print('CSV reports and run manifest:', result['output_dir'])
else:
    print('No new result yet; no model performance has been measured by this notebook.')


## Outputs and references

- Pretrained model/tokenizer: standard Hugging Face cache (usually `~/.cache/huggingface`). The first checkpoint download is on the order of hundreds of MB; installed PyTorch packages also use disk space.
- Token and inner/outer prediction caches: `data/processed/10_distilbert/<hash>/` (Git-ignored).
- Fold metrics, matched comparisons, exclusions and manifest: `reports/classifier_distilbert/<hash>/`.
- No fine-tuned model weights are retained. If this candidate is eventually selected, do a separate final-development fit and export for deployment.

All notebook output starts empty. No results are claimed before you execute training.

Sources: [DistilBERT documentation](https://huggingface.co/docs/transformers/v4.57.3/en/model_doc/distilbert), [PyTorch MPS](https://docs.pytorch.org/docs/main/notes/mps.html), [pinned pretrained checkpoint](https://huggingface.co/distilbert/distilbert-base-uncased/tree/12040accade4e8a0f71eabdb258fecc2e7e948be).
